# Check train/test leakage and evaluate by question/context group

Check whether different answers to the same question and context appear in both training and test data. Removing identical rows alone does not prevent this overlap.
We compare the existing random split with a grouped split. The grouped split keeps each question/context pair on one side. Scores from different test sets are descriptive, not a controlled model comparison. This notebook does not modify the existing notebooks or dataset.


In [1]:
from pathlib import Path
import re
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

root = Path.cwd()
if not (root / "data").exists() and (root.parent / "data").exists():
    root = root.parent
raw = pd.read_json(root / "data/llm_hallucination_detection_dataset.jsonl", lines=True)
required = ["prompt", "context", "response", "label"]
assert raw[required].notna().all().all(), "Missing required values: inspect data first."
assert set(raw["label"]) == {"faithful", "hallucinated"}
conflicts = raw.groupby(["prompt", "context", "response"])["label"].nunique()
assert conflicts.max() == 1, "Identical inputs have conflicting labels."
df = raw.drop_duplicates(["prompt", "context", "response"]).reset_index(drop=True)
# A group represents one exact question/context pair, including all its answers.
groups = df.groupby(["prompt", "context"], sort=False).ngroup()
print(f"Original rows: {len(raw)}; unique examples: {len(df)}; groups: {groups.nunique()}")
print(df["label"].value_counts())


Original rows: 5000; unique examples: 192; groups: 96
label
faithful        96
hallucinated    96
Name: count, dtype: int64


## 1. Audit the existing split

This reproduces the original random split (20% test, seed 42). A shared group means the model may have seen another answer to the same question and evidence during training. It is a potential source of optimistic evaluation, not proof that a model memorized the data.


In [2]:
random_train, random_test = train_test_split(df.index, test_size=0.2, random_state=42)
shared = set(groups.loc[random_train]) & set(groups.loc[random_test])
print("Question/context groups shared by train and test:", len(shared))
print("Test examples whose group occurs in training:", int(groups.loc[random_test].isin(shared).sum()))


Question/context groups shared by train and test: 33
Test examples whose group occurs in training: 33


## 2. Hold out complete groups

The test size below refers to groups; the fraction of examples may differ. We use one fixed seed and do not choose a split based on model performance. Exact grouping does not detect paraphrased or otherwise related contexts.


In [3]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
group_train, group_test = next(splitter.split(df, groups=groups))
assert set(groups.iloc[group_train]).isdisjoint(set(groups.iloc[group_test]))
for name, indices in [("Grouped train", group_train), ("Grouped test", group_test)]:
    print(name, "examples:", len(indices))
    print(df.iloc[indices]["label"].value_counts())
    assert df.iloc[indices]["label"].nunique() == 2, "Both classes are needed for this comparison."


Grouped train examples: 152
label
faithful        76
hallucinated    76
Name: count, dtype: int64
Grouped test examples: 40
label
faithful        20
hallucinated    20
Name: count, dtype: int64


## 3. Compare the existing baselines

The overlap threshold stays fixed at 0.6. TF-IDF is fitted only on training data. Label-derived fields such as factuality scores and hallucination type are excluded. These experiments do not retrain BERT or RoBERTa.


In [4]:
def words(text):
    return set(re.findall(r"[a-z0-9]+", str(text).lower()))

def overlap(response, context):
    answer_words = words(response)
    return len(answer_words & words(context)) / len(answer_words) if answer_words else 0.0

def features(frame, novel=False):
    if novel:
        return pd.Series([" ".join(sorted(words(r) - words(c)))
                          for r, c in zip(frame["response"], frame["context"])], index=frame.index)
    return frame["prompt"] + " " + frame["context"] + " " + frame["response"]

rows = []
errors = []
for split_name, train_idx, test_idx in [
    ("Original random", random_train, random_test),
    ("Grouped", group_train, group_test),
]:
    train, test = df.loc[train_idx], df.loc[test_idx]
    predictions = {"Word overlap": ["hallucinated" if overlap(r, c) < 0.6 else "faithful"
                                    for r, c in zip(test["response"], test["context"])]}
    for model_name, novel in [("TF-IDF full text", False), ("TF-IDF novel words", True)]:
        model = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000, random_state=42))
        model.fit(features(train, novel), train["label"])
        predictions[model_name] = model.predict(features(test, novel))
    for model_name, predicted in predictions.items():
        rows.append({"split": split_name, "model": model_name, "test_examples": len(test),
                     "accuracy": accuracy_score(test["label"], predicted),
                     "macro_f1": f1_score(test["label"], predicted, average="macro", zero_division=0)})
        if split_name == "Grouped":
            print("\n", model_name)
            print(classification_report(test["label"], predicted, digits=3, zero_division=0))
            print("Confusion matrix; rows=true, columns=predicted; order: faithful, hallucinated")
            print(confusion_matrix(test["label"], predicted, labels=["faithful", "hallucinated"]))
            detail = test[required].copy()
            detail["prediction"] = list(predicted)
            detail["model"] = model_name
            errors.append(detail.loc[detail["label"] != detail["prediction"]])
results = pd.DataFrame(rows)
display(results.round(4))



 Word overlap
              precision    recall  f1-score   support

    faithful      0.800     1.000     0.889        20
hallucinated      1.000     0.750     0.857        20

    accuracy                          0.875        40
   macro avg      0.900     0.875     0.873        40
weighted avg      0.900     0.875     0.873        40

Confusion matrix; rows=true, columns=predicted; order: faithful, hallucinated
[[20  0]
 [ 5 15]]

 TF-IDF full text
              precision    recall  f1-score   support

    faithful      1.000     0.900     0.947        20
hallucinated      0.909     1.000     0.952        20

    accuracy                          0.950        40
   macro avg      0.955     0.950     0.950        40
weighted avg      0.955     0.950     0.950        40

Confusion matrix; rows=true, columns=predicted; order: faithful, hallucinated
[[18  2]
 [ 0 20]]

 TF-IDF novel words
              precision    recall  f1-score   support

    faithful      1.000     1.000     1.00

,split,model,test_examples,accuracy,macro_f1
0,Original random,Word overlap,39,0.9744,0.9743
1,Original random,TF-IDF full text,39,0.5385,0.5307
2,Original random,TF-IDF novel words,39,0.9744,0.9743
3,Grouped,Word overlap,40,0.8750,0.8730
4,Grouped,TF-IDF full text,40,0.9500,0.9499
5,Grouped,TF-IDF novel words,40,1.0000,1.0000


## 4. Inspect errors and record findings

Read the context and answer for each mistake. Identify whether the answer changes an entity, number, date, negation, or adds an unsupported claim. Do not change the method using these test errors and then report the same test as an untouched final evaluation.


In [5]:
mistakes = pd.concat(errors, ignore_index=True)
pd.set_option("display.max_colwidth", 180)
display(mistakes)


,prompt,context,response,label,prediction,model
0,At what temperature does water boil at standard atmospheric pressure?,Water boils at 100°C at standard atmospheric pressure.,Water boils at 120°C at standard atmospheric pressure.,hallucinated,faithful,Word overlap
1,Check the factual accuracy of this answer: In what year did World War II end?,World War II ended in 1945.,World War II ended in 1952.,hallucinated,faithful,Word overlap
2,Provide a grounded response to the question: When was the Declaration of Independence adopted?,"The Declaration of Independence was adopted by the Continental Congress on July 4, 1776.","It was adopted on July 4, 1789.",hallucinated,faithful,Word overlap
3,Answer using only the provided context: What is the capital of Japan?,Tokyo is the capital city of Japan.,Kyoto is currently the capital city of Japan.,hallucinated,faithful,Word overlap
4,Check the factual accuracy of this answer: At what temperature does water boil at standard atmospheric pressure?,Water boils at 100°C at standard atmospheric pressure.,Water boils at 120°C at standard atmospheric pressure.,hallucinated,faithful,Word overlap
5,At what temperature does water boil at standard atmospheric pressure?,Water boils at 100°C at standard atmospheric pressure.,Water boils at 100°C at standard atmospheric pressure.,faithful,hallucinated,TF-IDF full text
6,Check the factual accuracy of this answer: At what temperature does water boil at standard atmospheric pressure?,Water boils at 100°C at standard atmospheric pressure.,Water boils at 100°C at standard atmospheric pressure.,faithful,hallucinated,TF-IDF full text


## Findings

### Train/test overlap
After removing duplicate examples, the dataset contained 192 examples in 96 question/context groups. In the original random split, 33 groups appeared in both training and test data. This affected 33 of the 39 test examples and creates a risk of optimistic evaluation.

The grouped split used 152 training examples and 40 test examples, with no exact question/context groups shared between them. Both sets were balanced between faithful and hallucinated answers.

### Model performance
On the grouped test set:

| Model | Accuracy | Macro F1 |
| --- | --- | --- |
| Word overlap | 87.50% | 0.8730 |
| TF-IDF full text + Logistic Regression | 95.00% | 0.9499 |
| TF-IDF novel words + Logistic Regression | 100.00% | 1.0000 |

The novel-words model performed best, correctly classifying all 40 test examples. It uses words present in the response but absent from the context.

### Error analysis
The word-overlap method incorrectly classified five hallucinated answers as faithful. These answers retained much of the context's wording while changing an important fact. For example:

- The context stated that water boils at 100°C, but the response changed the temperature to 120°C.
- The context identified Tokyo as Japan's capital, but the response named Kyoto.

The full-text TF-IDF model incorrectly classified two faithful answers about water boiling at 100°C as hallucinated. The novel-words model made no errors on this test set.

### Limitations
The dataset is small and synthetic, and these results come from one split. The perfect score does not demonstrate reliable detection on real LLM outputs.

Exact question/context grouping prevents identical pairs from crossing the split, but differently worded questions about the same facts may still appear in both sets. The random and grouped splits also contain different test examples, so score differences cannot be attributed solely to removing group overlap.
